# 01 · CICIDS2017 audit + leakage-safe preprocessing
Run first. Reads the 8 raw CSVs, fixes the data problems found in the earlier audit, builds the Mon–Wed / Thu / Fri splits, fits cleaning statistics on **train only**, and saves everything to Drive.

**Fixes versus the earlier audit run**
1. `pd.read_csv(usecols=[label, ts])` returns columns in *file* order (Timestamp before Label), so the old code swapped them: that is why labels looked like timestamps and Monday showed 25,998 "labels".
2. `Thursday-WorkingHours-Morning-WebAttacks.csv` has **288,602 fully blank trailing rows**. They are dropped here (they inflated the NaN counts and the duplicate count; 3,119,345 − 288,602 = 2,830,743 real flows).
3. Timestamps are **day-first** (`03/07/2017` = 3 July, `5/7/2017` = 5 July), not month-first. The "critical correction" was wrong, which is why Mon–Thu failed the date check.
4. Timestamps use a 12-hour clock with no AM/PM (afternoon shows as `3:30`). Hours 1–7 are shifted +12 h so chronological sorting is correct.
5. The duplicated `Fwd Header Length` column (`.1`) is dropped, and the `Web Attack � …` label encoding is normalised.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/NIDS_CICIDS2017_Project')
RAW, EXTRACT, ART, PROC, REP, MODELS = [ROOT/d for d in ['raw','extracted_generated_flows','artifacts','processed','reports','models']]
for d in [RAW, EXTRACT, ART, PROC, REP, MODELS]: d.mkdir(parents=True, exist_ok=True)
SEED = 42

import re, json, zipfile
import numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)

Mounted at /content/drive


In [2]:
# Locate the 8 CSVs (extract the ZIP from Drive if needed; nothing is re-uploaded).
csvs = sorted(EXTRACT.rglob('*.csv'))
if len(csvs) != 8:
    zips = list(ROOT.rglob('GeneratedLabelledFlows*.zip'))
    assert zips, f'Put GeneratedLabelledFlows.zip inside {RAW}'
    with zipfile.ZipFile(zips[0]) as z: z.extractall(EXTRACT)
    csvs = sorted(EXTRACT.rglob('*.csv'))
assert len(csvs) == 8, f'Expected 8 CSVs, found {len(csvs)}'
for p in csvs: print(p.name)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [3]:
DAYS = ['Monday','Tuesday','Wednesday','Thursday','Friday']
EXPECTED_DATE = {'Monday':'2017-07-03','Tuesday':'2017-07-04','Wednesday':'2017-07-05','Thursday':'2017-07-06','Friday':'2017-07-07'}
META = ['Flow ID','Source IP','Source Port','Destination IP','Destination Port','Protocol','Timestamp','Label']  # never model inputs

def clean_label(s):
    return s.astype(str).str.strip().str.replace(r'[^\x20-\x7E]+', '-', regex=True)   # 'Web Attack � XSS' -> 'Web Attack - XSS'

def parse_ts(s):
    s = s.astype(str).str.strip()
    t = pd.to_datetime(s, format='%d/%m/%Y %H:%M', errors='coerce')
    miss = t.isna()
    if miss.any():
        t[miss] = pd.to_datetime(s[miss], dayfirst=True, errors='coerce', format='mixed')
    return t.where(t.dt.hour >= 8, t + pd.Timedelta(hours=12))   # capture ran ~08:00-17:00 on a 12h clock

frames, audit, FEATS = [], [], None
for p in csvs:
    day = next(d for d in DAYS if d.lower() in p.name.lower())
    df = pd.read_csv(p, low_memory=False, encoding_errors='replace')
    df.columns = [re.sub(r'\s+', ' ', str(c).strip()) for c in df.columns]
    n_raw = len(df)
    df = df.drop(columns=[c for c in df.columns if c == 'Fwd Header Length.1'])
    df = df.dropna(subset=['Label'])                       # blank trailing rows
    feat = [c for c in df.columns if c not in META]
    if FEATS is None: FEATS = feat
    X = df[FEATS].apply(pd.to_numeric, errors='coerce')
    n_inf = int(np.isinf(X.to_numpy(dtype='float64')).sum())
    X = X.replace([np.inf, -np.inf], np.nan).astype('float32').reset_index(drop=True)
    X['Timestamp'] = parse_ts(df['Timestamp']).reset_index(drop=True)
    X['Label'] = clean_label(df['Label']).reset_index(drop=True)
    X['day'] = day
    dates = sorted(set(map(str, X['Timestamp'].dt.date.dropna().unique())))
    audit.append({'file': p.name, 'day': day, 'rows_raw': n_raw, 'blank_rows_dropped': n_raw - len(df), 'rows': len(X),
                  'n_labels': X['Label'].nunique(), 'inf_cells': n_inf, 'nan_cells(after inf->nan)': int(X[FEATS].isna().sum().sum()),
                  'NaT_timestamps': int(X['Timestamp'].isna().sum()), 'dates': ','.join(dates), 'date_ok': dates == [EXPECTED_DATE[day]]})
    frames.append(X)
audit_df = pd.DataFrame(audit); display(audit_df)
audit_df.to_csv(REP/'file_audit.csv', index=False)
print('Features before correlation/constant filtering:', len(FEATS), '| total real rows:', f"{audit_df.rows.sum():,}")
assert audit_df.date_ok.all() and audit_df.NaT_timestamps.sum() == 0, 'Timestamp/date check failed - inspect audit table'

,file,day,rows_raw,blank_rows_dropped,rows,n_labels,inf_cells,nan_cells(after inf->nan),NaT_timestamps,dates,date_ok
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday,225745,0,225745,2,64,68,0,2017-07-07,True
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday,286467,0,286467,2,727,742,0,2017-07-07,True
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday,191033,0,191033,2,216,244,0,2017-07-07,True
3,Monday-WorkingHours.pcap_ISCX.csv,Monday,529918,0,529918,1,810,874,0,2017-07-03,True
4,Thursday-WorkingHours-Afternoon-Infilteration....,Thursday,288602,0,288602,2,396,414,0,2017-07-06,True
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday,458968,288602,170366,4,250,270,0,2017-07-06,True
6,Tuesday-WorkingHours.pcap_ISCX.csv,Tuesday,445909,0,445909,3,327,528,0,2017-07-04,True
7,Wednesday-workingHours.pcap_ISCX.csv,Wednesday,692703,0,692703,6,1586,2594,0,2017-07-05,True


Features before correlation/constant filtering: 76 | total real rows: 2,830,743


In [4]:
data = pd.concat(frames, ignore_index=True); del frames
data['day_id'] = data['day'].map({d: i for i, d in enumerate(DAYS)}).astype('int8')
order = np.lexsort((np.arange(len(data)), data['Timestamp'].values.astype('int64'), data['day_id'].values))   # day, time, original order
data = data.iloc[order].reset_index(drop=True)
print(data.groupby('day')['Timestamp'].agg(['min','max','count']))

lab_by_day = pd.crosstab(data['Label'], data['day'])[DAYS]
lab_by_day.to_csv(REP/'label_counts_by_day.csv'); display(lab_by_day)

SPLITS = {'train': ['Monday','Tuesday','Wednesday'], 'val': ['Thursday'], 'test': ['Friday']}
part = {k: data[data['day'].isin(v)].reset_index(drop=True) for k, v in SPLITS.items()}; del data
for k, v in part.items(): print(k, f'{len(v):,} rows')

                          min                 max   count
day                                                      
Friday    2017-07-07 08:59:00 2017-07-07 17:02:00  703245
Monday    2017-07-03 08:55:58 2017-07-03 17:01:34  529918
Thursday  2017-07-06 08:59:00 2017-07-06 17:04:00  458968
Tuesday   2017-07-04 08:53:00 2017-07-04 17:00:00  445909
Wednesday 2017-07-05 08:42:00 2017-07-05 17:10:00  692703


day,Monday,Tuesday,Wednesday,Thursday,Friday
Label,,,,,
BENIGN,529918,432074,440031,456752,414322
Bot,0,0,0,0,1966
DDoS,0,0,0,0,128027
DoS GoldenEye,0,0,10293,0,0
DoS Hulk,0,0,231073,0,0
DoS Slowhttptest,0,0,5499,0,0
DoS slowloris,0,0,5796,0,0
FTP-Patator,0,7938,0,0,0
Heartbleed,0,0,11,0,0


train 1,668,530 rows
val 458,968 rows
test 703,245 rows


In [5]:
# Exact duplicates (features + Label). Removed from TRAIN only; val/test are left untouched (no test-day resampling).
dup_report = {}
for k, df in part.items():
    dup_report[k] = int(df.duplicated(subset=FEATS + ['Label']).sum())
print('Exact duplicate rows per split:', dup_report)
tr = part['train']; part['train'] = tr[~tr.duplicated(subset=FEATS + ['Label'])].reset_index(drop=True); del tr
print('Train rows after de-duplication:', f"{len(part['train']):,}")

# Fit-on-train-only statistics
tr = part['train']
medians = tr[FEATS].median()
imp = tr[FEATS].fillna(medians)
nun = imp.nunique()
constant = nun[nun <= 1].index.tolist()
remaining = [c for c in FEATS if c not in constant]
corr = imp[remaining].sample(min(500_000, len(imp)), random_state=SEED).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
correlated = [c for c in upper.columns if (upper[c] > 0.95).any()]
final = [c for c in remaining if c not in correlated]
del imp, corr, upper
print(f'constant dropped: {len(constant)} | correlated (>0.95) dropped: {len(correlated)} | kept: {len(final)}')
print('constant:', constant); print('correlated:', correlated)

Exact duplicate rows per split: {'train': 270132, 'val': 102566, 'test': 195917}
Train rows after de-duplication: 1,398,398
constant dropped: 10 | correlated (>0.95) dropped: 22 | kept: 44
constant: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
correlated: ['Total Backward Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Std', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Fwd Packets/s', 'Max Packet Length', 'Packet Length Std', 'SYN Flag Count', 'ECE Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Idle Mean', 'Idle Max', 'Idle Min']


In [6]:
# Apply train medians to every split and save to Drive.
for k, df in part.items():
    out = pd.concat([df[final].fillna(medians[final]).astype('float32'), df[['Timestamp','Label','day','day_id']]], axis=1)
    out.to_parquet(PROC/f'{k}.parquet', index=False)
    print('saved', k, out.shape)

# Label vocabulary: BENIGN first, then the other classes seen in TRAIN ('known'); everything else is unseen at training time.
all_labels = sorted(set().union(*[set(df['Label'].unique()) for df in part.values()]))
known = ['BENIGN'] + sorted(l for l in part['train']['Label'].unique() if l != 'BENIGN')
glob = {l: i for i, l in enumerate(['BENIGN'] + sorted(l for l in all_labels if l != 'BENIGN'))}
(ART/'label_mapping.json').write_text(json.dumps({'global': glob, 'known': known}, indent=2))
(ART/'preprocessing_meta.json').write_text(json.dumps({
    'features': final, 'constant_dropped': constant, 'correlated_dropped': correlated,
    'train_medians': {c: float(medians[c]) for c in FEATS}, 'duplicates_by_split': dup_report,
    'splits': SPLITS, 'seed': SEED, 'timestamp': 'day-first, +12h for hours<8', 'excluded_metadata': META}, indent=2))

summary = pd.DataFrame({k: df['Label'].value_counts() for k, df in part.items()}).fillna(0).astype(int)
summary['in_train'] = summary.index.isin(known); display(summary)
print('Known (trained) classes:', known)
print('Unseen at training time:', [l for l in all_labels if l not in known])

saved train (1398398, 48)
saved val (458968, 48)
saved test (703245, 48)


,train,val,test,in_train
Label,,,,
BENIGN,1195502,456752,414322,True
Bot,0,0,1966,False
DDoS,0,0,128027,False
DoS GoldenEye,10282,0,0,True
DoS Hulk,172849,0,0,True
DoS Slowhttptest,5228,0,0,True
DoS slowloris,5374,0,0,True
FTP-Patator,5933,0,0,True
Heartbleed,11,0,0,True


Known (trained) classes: ['BENIGN', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'SSH-Patator']
Unseen at training time: ['Bot', 'DDoS', 'Infiltration', 'PortScan', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']
